# Lábia — qual configuração de treino é rápida o bastante?

O teste de fumaça do treino mediu **154 tokens por segundo** com QLoRA (modelo em 4 bits) numa T4. Nesse ritmo, uma passada por mil conversas de ~3 mil tokens leva 6,5 horas, e duas passadas não cabem numa sessão de 12 horas.

Aqui medimos três alternativas, cada uma num processo separado, com os mesmos dados e o mesmo número de passos:

| Configuração | Ideia |
|---|---|
| LoRA em 16 bits, 1 GPU | sem o custo de descomprimir os pesos de 4 bits a cada passo |
| LoRA em 16 bits, 2 GPUs | **paralelismo de dados**: cada GPU processa metade das conversas do lote e as duas somam o que aprenderam |
| Unsloth (4 e 16 bits), 1 GPU | biblioteca com kernels otimizados para este tipo de treino em T4 |

Os dados são do agente roteirizado: só servem para medir velocidade e memória.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
LABIA_COMMIT = "4e001af61eeff09d1946e9f16c64f2c6a5b0404e"
STEPS = 8
WORK = Path("/kaggle/working/velocidade")
WORK.mkdir(parents=True, exist_ok=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes",
                    f"git+https://github.com/deniscarvalho-tech/labia.git@{LABIA_COMMIT}"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-600:])

## Dados de formato real (agente roteirizado)

In [ ]:
from labia.agents import ScriptedAgent, ScriptedCustomer
from labia.conversation import run_conversation
from labia.dataset import to_sft_example
from labia.scenarios.generator import Counts, generate
import random
train = generate("train", Counts(train_per_cell=1))
random.Random(0).shuffle(train)
with open(WORK / "sft-teste.jsonl", "w", encoding="utf-8") as fh:
    for s in train[:96]:
        rec = run_conversation(s, ScriptedAgent(s, "ideal"), ScriptedCustomer(s), model_id="roteirizado")
        fh.write(json.dumps(to_sft_example(rec, s), ensure_ascii=False) + "\n")
print("ok")

## Rodar cada configuração num processo separado

In [ ]:
results = {}


def run(name, cmd, env_extra=None):
    out = WORK / name
    env = dict(os.environ, **(env_extra or {}))
    start = time.perf_counter()
    p = subprocess.run(cmd + ["--data", str(WORK / "sft-teste.jsonl"), "--out", str(out), "--max-steps", str(STEPS),
                              "--grad-accum", "4"], capture_output=True, text=True, env=env)
    wall = time.perf_counter() - start
    stats_file = out / "treino.json"
    if p.returncode != 0 or not stats_file.exists():
        print(f"[{name}] FALHOU em {wall:.0f} s\n", (p.stdout + p.stderr)[-2500:])
        results[name] = {"falhou": True, "erro": (p.stdout + p.stderr)[-1500:]}
        return
    s = json.loads(stats_file.read_text(encoding="utf-8"))
    s.pop("log", None)
    s["segundos_total_com_carga"] = round(wall, 1)
    results[name] = s
    print(f"[{name}] {s['tokens_por_segundo']} tokens/s, pico {s['pico_memoria_gb']} GB, {wall:.0f} s no total")


one_gpu = {"CUDA_VISIBLE_DEVICES": "0"}
run("lora16-1gpu", [sys.executable, "-m", "labia.train_sft", "--mode", "lora16"], one_gpu)
run("lora16-2gpus", ["torchrun", "--nproc_per_node", "2", "-m", "labia.train_sft", "--mode", "lora16"])
run("qlora-2gpus", ["torchrun", "--nproc_per_node", "2", "-m", "labia.train_sft", "--mode", "qlora"])

## Unsloth (por último, porque a instalação pode trocar versões de bibliotecas)

In [ ]:
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "unsloth"], capture_output=True, text=True)
print((r.stdout + r.stderr)[-1200:])
run("unsloth-qlora-1gpu", [sys.executable, "-m", "labia.train_sft", "--unsloth", "--mode", "qlora"], one_gpu)
run("unsloth-lora16-1gpu", [sys.executable, "-m", "labia.train_sft", "--unsloth", "--mode", "lora16"], one_gpu)

## Resumo

In [ ]:
(WORK / "resumo.json").write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"{'configuração':24} {'tokens/s':>9} {'memória':>8}")
print(f"{'qlora-1gpu (medido antes)':24} {154:9.0f} {11.15:8.2f}")
for name, s in results.items():
    if s.get("falhou"):
        print(f"{name:24} {'falhou':>9}")
    else:
        print(f"{name:24} {s['tokens_por_segundo']:9.0f} {s['pico_memoria_gb']:8.2f}")